# TS2TabPFN: ejecución en Kaggle

Tercera sesión de apoyo a las dos de Colab (`docs/ejecucion.md`, sección Kaggle). Corre los datasets que se indiquen en `DATASETS` y deja los resultados en `/kaggle/working`, en el mismo formato que Colab (`results/<extractor>/<dataset>/resample_XX.json`), para subirlos después a Drive.

Antes de la primera vez:
1. *Settings → Accelerator* → **GPU T4 x2** (o P100) y *Settings → Turn on internet*.
2. *Add-ons → Secrets* → agrega **`HF_TOKEN`** (el mismo token de Hugging Face de Colab) y actívalo para este notebook.

Para que corra en segundo plano: **Save Version → Save & Run All (Commit)**. Kaggle corta a las 12 h y en ese caso no guarda nada; por eso la celda de ejecución se detiene sola a las `TIME_BUDGET_H` horas y guarda lo terminado.

In [ ]:
# === Parámetros de esta corrida ===
EXTRACTOR = "multirocket"          # "c22", "tsfresh" o "multirocket" (tsfresh necesitaría las features como input)
DATASETS = ["PhonemeSpectra"]      # lista explícita; se corren del más barato al más caro
RESAMPLES = "0-29"
TIME_BUDGET_H = 10.5               # Kaggle corta a las 12 h; deja margen para comprimir y guardar

BRANCH = "phase-1"
REPO_URL = "https://github.com/JEFRY200307/ProyectoDeepSkills.git"
TSC_ZIP_ID = "1p8D-haDDwPKloKMBQCQHgENYHVsjixR4"    # references/TS2TabPFN/README.md

WORK = "/kaggle/working"           # se guarda como output de la versión
TMP = "/kaggle/temp"               # disco de la sesión, no se guarda
RESULTS_DIR = f"{WORK}/results"

In [ ]:
# Código del proyecto (rama BRANCH)
import os
os.makedirs(TMP, exist_ok=True)
%cd {TMP}
![ -d repo ] || git clone -q -b {BRANCH} {REPO_URL} repo
!git -C repo fetch -q origin {BRANCH} && git -C repo checkout -q {BRANCH} && git -C repo reset -q --hard origin/{BRANCH}
!git -C repo log --oneline -1
!nproc; free -g | head -2; nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Entorno: Python 3.10 con las versiones exactas del paper (decisión #9). Tarda unos minutos.
!pip install -q uv==0.9.5
![ -x {TMP}/venv/bin/python ] || uv venv -q --python 3.10 {TMP}/venv
!uv pip install -q --python {TMP}/venv/bin/python -r {TMP}/repo/requirements.txt
!{TMP}/venv/bin/python -c "import tabpfn, aeon, torch; print('tabpfn', tabpfn.__version__, '| aeon', aeon.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
# Token de Hugging Face para los pesos de TabPFN 2.5 (desde Kaggle Secrets; nunca se escribe en el notebook)
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN cargado:", bool(os.environ["HF_TOKEN"]))
os.environ["TABPFN_MODEL_CACHE_DIR"] = f"{TMP}/models"
os.makedirs(os.environ["TABPFN_MODEL_CACHE_DIR"], exist_ok=True)

In [ ]:
# Datos: el zip de los autores (1.3 GB) se descarga en cada corrida
ZIP = f"{TMP}/tsc_data.zip"
if not os.path.exists(ZIP):
    !{TMP}/venv/bin/python -m gdown {TSC_ZIP_ID} -O {ZIP}
if not os.path.exists(f"{TMP}/data_tsc"):
    !unzip -q -o {ZIP} -d {TMP}/data_tsc

import glob
hits = glob.glob(f"{TMP}/data_tsc/**/ECG200/ECG200_TRAIN.ts", recursive=True)
DATA_DIR = os.path.dirname(os.path.dirname(hits[0]))
print("DATA_DIR =", DATA_DIR, "|", len(os.listdir(DATA_DIR)), "carpetas")

In [ ]:
# Resultados de una corrida anterior de Kaggle (opcional): si agregaste su output como Input,
# se copian aquí y run_tsc.py salta esos resamples.
import shutil
for prev in glob.glob("/kaggle/input/**/results", recursive=True):
    print("copiando resultados previos de", prev)
    shutil.copytree(prev, RESULTS_DIR, dirs_exist_ok=True)
print(len(glob.glob(f"{RESULTS_DIR}/*/*/resample_*.json")), "resamples ya hechos")

In [ ]:
# Ejecución, con tope de tiempo: a las TIME_BUDGET_H horas se detiene y conserva lo terminado
# (cada resample se escribe de forma atómica, así que nunca queda uno a medias).
import shlex, subprocess, sys, threading, time

cmd = [f"{TMP}/venv/bin/python", "-u", "-X", "faulthandler", f"{TMP}/repo/scripts/run_tsc.py",
       "--extractor", EXTRACTOR, "--shard", "0", "--num-shards", "1",
       "--resamples", RESAMPLES,
       "--data-dir", DATA_DIR,
       "--features-dir", f"{TMP}/features",
       "--results-dir", RESULTS_DIR,
       "--datasets", *DATASETS]
if EXTRACTOR == "tsfresh":
    cmd += ["--no-extract"]

os.makedirs(f"{WORK}/logs", exist_ok=True)
log_path = f"{WORK}/logs/{EXTRACTOR}_kaggle_{time.strftime('%Y%m%d_%H%M%S')}.log"
print(" ".join(shlex.quote(c) for c in cmd), "\nlog:", log_path)
with open(log_path, "w") as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timer = threading.Timer(TIME_BUDGET_H * 3600, proc.kill)
    timer.start()
    for line in proc.stdout:
        sys.stdout.write(line)
        log.write(line)
        log.flush()
    code_ = proc.wait()
    timer.cancel()
print("exit code:", code_, "(-9 = detenido por TIME_BUDGET_H; lo terminado está guardado)")

In [ ]:
# Empaqueta los resultados para subirlos a Drive (descomprimir en MyDrive/ts2tabpfn/)
done = sorted(glob.glob(f"{RESULTS_DIR}/*/*/resample_*.json"))
print(len(done), "resamples en esta corrida")
for d in sorted({p.split("/")[-2] for p in done}):
    print(" ", d, sum(1 for p in done if p.split("/")[-2] == d), "/ 30")
stamp = time.strftime("%Y%m%d_%H%M%S")
!cd {WORK} && zip -q -r results_kaggle_{stamp}.zip results logs && ls -lh {WORK}/*.zip